In [2]:
import os
api_key=os.getenv("API_KEY")
model_name="gemini-3.1-flash-lite-preview"

In [ ]:
# load PDFs

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


documents_dir = Path("../../data/sec10k")
print(documents_dir)
if not documents_dir.is_dir():
    raise FileNotFoundError(f"Documents folder not found: {documents_dir.resolve()}")

pdf_paths = sorted(
    (path for path in documents_dir.rglob("*") if path.is_file() and path.suffix.lower() == ".pdf"),
    key=lambda path: path.as_posix().lower(),
)
if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {documents_dir.resolve()}")

documents = []
for pdf_path in pdf_paths:
    loaded_documents = PyPDFLoader(str(pdf_path), mode="single").load()
    for document in loaded_documents:
        document.metadata["source"] = str(pdf_path)
        document.metadata["file_name"] = pdf_path.name
    documents.extend(loaded_documents)

print(f"Loaded {len(documents)} PDFs as Document objects")

In [ ]:
# text chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter
import json

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(documents)
chunk_dicts = [chunk.model_dump() for chunk in chunks] # Convert the list of Document objects into a list of dictionaries
print(f"Split into {len(chunks)} chunks")

with open("rp01_chunks.json", "w", encoding="utf-8") as file:
    json.dump(
        {"chunks": chunk_dicts},
        file,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved {len(chunks)} chunks to rp01_chunks.json")

In [ ]:
# create embeddings using openai text-embedding-3-small model

# from langchain_huggingface import HuggingFaceEmbeddings
from tqdm.auto import tqdm
from openai import OpenAI

# Initialize the client. It automatically picks up the OPENAI_API_KEY environment variable.
client = OpenAI(base_url="https://proxy.vectorinstitute.ai/v1",api_key=api_key)

vectors = []
for chunk in tqdm(chunks, desc="Embedding chunks", unit="chunk"):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk.page_content,
    ) 
    embedding_vector = response.data[0].embedding
    vectors.append(embedding_vector)

with open("rp01_embeddings.json", "w", encoding="utf-8") as file:
    json.dump(
        [
            {
                "metadata": chunk.metadata,
                "embedding": vector,
            }
            for chunk, vector in zip(chunks, vectors)
        ],
        file,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved {len(vectors)} embeddings to rp01_embeddings.json") 

In [3]:
# test gemini-3.1-flash-lite-preview model 

import os

from dotenv import load_dotenv
from openai import OpenAI


# Loads your environment variables from the .env file
load_dotenv()
client = OpenAI(base_url="https://proxy.vectorinstitute.ai/v1", api_key=api_key)

stream = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[{"role": "user", "content": "Hi, how are you?"}],
    stream=True,
)

for chunk in stream:
    if chunk.choices:
        print(chunk.choices[0].delta.content or "", end="\n", flush=True)

print("<--DEFAULT MODELS-->")
print(f"GENERATOR_MODEL{os.getenv('GENERATOR_MODEL')}")
print(f"AGENT_MODEL{os.getenv('AGENT_MODEL')}")
print(f"USER_SIMULATOR_MODEL{os.getenv('USER_SIMULATOR_MODEL')}")


I'm doing great, thank you for asking! How are you doing today?
 Is there anything I can help you with?

<--DEFAULT MODELS-->
GENERATOR_MODELNone
AGENT_MODELNone
USER_SIMULATOR_MODELNone


In [ ]:
# create vector store using Chroma

from langchain_chroma import Chroma

CLEAR_VECTOR_STORE = True

vector_store = Chroma(
    collection_name="rag_demo",
    # embedding_function=embeddings,
    persist_directory="./chroma_db",
)

if CLEAR_VECTOR_STORE:
    existing_ids = vector_store._collection.get(include=["metadatas"])["ids"]
    if existing_ids:
        vector_store.delete(ids=existing_ids)
    print("Cleared existing vector store")

vector_store.add_documents(documents=chunks)
print(f"Indexed {len(chunks)} chunks into Chroma vector store")

vectorstore_data = vector_store._collection.get(
    include=["documents", "metadatas", "embeddings"]
)
vectorstore_data["embeddings"] = vectorstore_data["embeddings"].tolist()

with open("rp01_vectorstore.json", "w", encoding="utf-8") as file:
    json.dump(vectorstore_data, file, ensure_ascii=False, indent=2)

print("Saved vector store to rp01_vectorstore.json")

In [ ]:
# create hybrid retriever using Chroma

top_k = 5
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": top_k},
)
print(f"Retriever created (top-k={top_k})")